# FungiCLEF 2025 — Frozen-Backbone + TTA + Transductive Prototypes

**Task:** few-shot fungi ID — **2427 classes, ~3.2 images/class**, 999 test observations.

**Design.** With ~3 images/class, full fine-tuning destroys pretrained features
(a CAFormer fine-tune collapsed to ~0%). The robust recipe is **frozen
foundation-model embeddings + a non-parametric head**, with accuracy levers layered on:

| Lever | What | Effect (honest, obs-level) |
|---|---|---|
| **BioCLIP-2** frozen embeddings (`imageomics/bioclip-2`, 500px) | in-domain Tree-of-Life model | top-1 46.7% / top-10 75.4% |
| **hflip TTA** | average original + flip embeddings | small, free |
| **train + val reference** | fold labelled val into the prototype pool for test | **+6–7 top-1, +5 top-10** |
| **soft transductive prototypes** | adapt centroids toward the test distribution (EM, soft votes) | **+1 top-5, +1.3 top-10** |
| Re-rankers (DINOv3, k-NN, metadata), caption text, hard pseudo-labels | tried, measured, dropped | no gain / hurt |

**Honest test estimate (observation-disjoint val CV): top-1 ≈ 53%, top-5 ≈ 75%, top-10 ≈ 83%.**
Real submission uses ALL of val as reference, so expect slightly higher.

In [1]:
import os, numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
from fungi_features import load_split_df, tta_embedding, extract, IMG_RES
from fungi_train import (build_metadata_features, build_experts, geo_combine,
                         cosine_centroid_probs, transductive_centroid_probs,
                         observation_topk, NUM_CLASSES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | {DEVICE} | res {IMG_RES} | classes {NUM_CLASSES}")

torch 2.9.1+cu128 | cuda | res 500p | classes 2427


## 1. Data & metadata overview

In [2]:
dfs = {sp: load_split_df(sp) for sp in ("train","val","test")}
y_train = dfs["train"]["category_id"].astype(int).values
y_val   = dfs["val"]["category_id"].astype(int).values
for sp,df in dfs.items():
    print(f"{sp:5s}: {len(df):5d} images | {df['observationID'].nunique():4d} observations")
counts = np.bincount(y_train, minlength=NUM_CLASSES)
print(f"\nclasses: min {counts.min()}, median {int(np.median(counts))}, max {counts.max()}, mean {counts.mean():.2f}/class")

train:  7819 images | 4293 observations
val  :  2285 images | 1099 observations
test :  1911 images |  999 observations

classes: min 1, median 2, max 30, mean 3.22/class


In [3]:
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].hist(counts,bins=range(0,counts.max()+2),color="#4C72B0")
ax[0].set(title="Images per class (few-shot)",xlabel="# images",ylabel="# classes")
dfs["train"]["month"].value_counts().sort_index().plot.bar(ax=ax[1],color="#55A868")
ax[1].set(title="Observations by month",xlabel="month")
dfs["train"]["habitat"].value_counts().head(10).plot.barh(ax=ax[2],color="#C44E52")
ax[2].set(title="Top-10 habitats"); ax[2].invert_yaxis()
plt.tight_layout(); plt.savefig("viz_ens_data_overview.png",dpi=110); plt.show()

## 2. Frozen embeddings + test-time augmentation
Each backbone runs on the original AND horizontally-flipped image; the L2-normalised
vectors are averaged (`tta_embedding`). All passes cached.

In [4]:
img = {sp:{bb:tta_embedding(bb,sp,DEVICE) for bb in ("bioclip2","dinov3")}
       for sp in ("train","val","test")}
def topk(p): return observation_topk(p,dfs["val"],labels=y_val)[0]
no_tta = cosine_centroid_probs(extract("bioclip2","train",DEVICE),y_train,extract("bioclip2","val",DEVICE))
tta    = cosine_centroid_probs(img["train"]["bioclip2"],y_train,img["val"]["bioclip2"])
print("BioCLIP-2 centroid  no-TTA:", topk(no_tta))
print("BioCLIP-2 centroid  +TTA  :", topk(tta))

BioCLIP-2 centroid  no-TTA: {'top1': 0.45950864422202004, 'top5': 0.6778889899909009, 'top10': 0.7552320291173794}
BioCLIP-2 centroid  +TTA  : {'top1': 0.46678798908098273, 'top5': 0.6833484986351228, 'top10': 0.7543221110100091}


## 3. Metadata encoding

In [5]:
meta = build_metadata_features(dfs)
def pack(sp): return {"bioclip2":img[sp]["bioclip2"],"dinov3":img[sp]["dinov3"],"meta":meta[sp]}
print("metadata dim:", meta["train"].shape[1])

metadata dim: 126


## 4. Experts (val, reference = train only)
BioCLIP-2 dominates; DINOv3 (general SSL) is far weaker on fungi; metadata weak alone.

In [6]:
freq=np.bincount(y_train,minlength=NUM_CLASSES).astype(float)
cw=np.where(freq>0,1/np.sqrt(freq),0.0); cw/=cw[freq>0].mean()
val_ex = build_experts(pack("train"), y_train, pack("val"), cw)
rows={n:observation_topk(p,dfs["val"],labels=y_val)[0] for n,p in val_ex.items()}
for n,m in rows.items(): print(f"{n:16s} top1={m['top1']:.4f} top5={m['top5']:.4f} top10={m['top10']:.4f}")
names=list(rows); x=np.arange(len(names)); w=0.25
fig,ax=plt.subplots(figsize=(9,4.5))
for i,k in enumerate(["top1","top5","top10"]): ax.bar(x+i*w,[rows[n][k] for n in names],w,label=k)
ax.set_xticks(x+w); ax.set_xticklabels(names,rotation=15); ax.legend()
ax.set(title="Expert comparison (val, train-only ref)",ylabel="obs-level accuracy")
plt.tight_layout(); plt.savefig("viz_ens_experts.png",dpi=110); plt.show()

cent_bioclip2    top1=0.4668 top5=0.6833 top10=0.7543
cent_dinov3      top1=0.1711 top5=0.3376 top10=0.4040
knn_bioclip2     top1=0.4058 top5=0.6497 top10=0.6752
meta             top1=0.0373 top5=0.1056 top10=0.1665


## 5. Two decisive levers: train+val reference & soft transduction

Ensemble weights are grid-searched on val (balanced top1/5/10). Then for **test** we
(a) fold the entire labelled **val** set into the reference pool, and (b) apply **soft
transductive prototype adaptation** — a few EM steps that nudge each class centroid
toward the unlabelled test distribution using probability-weighted votes (no hard
pseudo-labels, so wrong guesses can't snowball). We estimate the honest gain with an
**observation-disjoint** 2-fold CV on val.

In [7]:
def score(m): return (m["top1"]+m["top5"]+m["top10"])/3
best={"_s":-1}
for wk in (0.0,0.3,0.5,1.0):
  for wd in (0.0,0.1,0.2):
    for wm in (0.0,0.1,0.2,0.3,0.5):
      ww={"cent_bioclip2":1.0,"knn_bioclip2":wk,"cent_dinov3":wd,"meta":wm}
      m=observation_topk(geo_combine(val_ex,ww),dfs["val"],labels=y_val)[0]
      if score(m)>best["_s"]: best={**m,"_s":score(m),"w":ww}
bw=best["w"]; print("best weights:",{k:v for k,v in bw.items() if v})

best weights: {'cent_bioclip2': 1.0}


In [8]:
obs=dfs["val"]["observationID"].values; uniq=pd.unique(obs)
order=np.random.RandomState(0).permutation(len(uniq)); setA=set(uniq[order[:len(uniq)//2]])
A=np.array([i for i,o in enumerate(obs) if o in setA]); B=np.array([i for i,o in enumerate(obs) if o not in setA])
ens={"top1":[],"top5":[],"top10":[]}; tr={"top1":[],"top5":[],"top10":[]}
for ev,rf in [(A,B),(B,A)]:
    ref={k:np.concatenate([pack("train")[k],pack("val")[k][rf]]) for k in pack("train")}; ry=np.concatenate([y_train,y_val[rf]])
    es={k:pack("val")[k][ev] for k in pack("val")}
    m =observation_topk(geo_combine(build_experts(ref,ry,es,cw),bw),dfs["val"].iloc[ev],labels=y_val[ev])[0]
    mt=observation_topk(transductive_centroid_probs(ref["bioclip2"],ry,es["bioclip2"]),dfs["val"].iloc[ev],labels=y_val[ev])[0]
    for k in ens: ens[k].append(m[k]); tr[k].append(mt[k])
print(f"train+val ref            : top1={np.mean(ens['top1']):.4f} top5={np.mean(ens['top5']):.4f} top10={np.mean(ens['top10']):.4f}")
print(f"train+val ref + transduct: top1={np.mean(tr['top1']):.4f} top5={np.mean(tr['top5']):.4f} top10={np.mean(tr['top10']):.4f}")
# progression bar: train-only -> train+val -> +transduction
g0=[best["top1"],best["top5"],best["top10"]]
g1=[np.mean(ens[k]) for k in ["top1","top5","top10"]]; g2=[np.mean(tr[k]) for k in ["top1","top5","top10"]]
xx=np.arange(3); fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar(xx-0.25,g0,0.25,label="train-only ref (val)")
ax.bar(xx,g1,0.25,label="train+val ref (est. test)")
ax.bar(xx+0.25,g2,0.25,label="+ transduction (est. test)")
ax.axhline(0.80,ls="--",c="grey",lw=1); ax.set_xticks(xx); ax.set_xticklabels(["top1","top5","top10"])
ax.set(title="Accuracy progression across levers",ylabel="accuracy"); ax.legend()
plt.tight_layout(); plt.savefig("viz_ens_reference_gain.png",dpi=110); plt.show()

train+val ref            : top1=0.5287 top5=0.7434 top10=0.8189
train+val ref + transduct: top1=0.5296 top5=0.7543 top10=0.8317


## 6. What did NOT help (measured, not assumed)
- **Caption text** (BioCLIP-2 text tower on per-image descriptions): top-1 ~3% alone,
  fusing *degrades* the image model — captions too generic for 2427 fine species.
- **Hard pseudo-labelling** (add top-confidence test obs with hard labels): ~flat —
  wrong labels cancel right ones. The **soft** version (§5) works instead.
- **DINOv3 / k-NN / metadata re-rankers**: grid drives weights to ~0 once the
  reference is strong. **Power transform** on embeddings: slightly hurts (signed CLIP feats).

## 7. Diagnostics

In [9]:
val_probs = geo_combine(val_ex, bw)
_,obs_ids,tk = observation_topk(val_probs,dfs["val"],ks=(1,))
olab={o:yv for o,yv in zip(dfs["val"]["observationID"].values,y_val)}
hit=np.zeros(NUM_CLASSES); tot=np.zeros(NUM_CLASSES)
for o,p in zip(obs_ids,tk[:,0]): c=olab[o]; tot[c]+=1; hit[c]+=(p==c)
pres=tot>0
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].hist((hit[pres]/tot[pres]),bins=20,color="#4C72B0")
ax[0].set(title=f"Per-class top-1 acc (val, {pres.sum()} classes)",xlabel="accuracy",ylabel="# classes")
bins=[0,1,2,3,5,10,9999]; lbl=["1","2","3","4-5","6-10","10+"]; ab=[]
for lo,hi in zip(bins[:-1],bins[1:]):
    s=pres&(counts>lo)&(counts<=hi); ab.append(np.nan if s.sum()==0 else hit[s].sum()/tot[s].sum())
ax[1].bar(lbl,ab,color="#55A868"); ax[1].set(title="Top-1 acc vs #train images/class",xlabel="train images",ylabel="accuracy")
plt.tight_layout(); plt.savefig("viz_ens_diagnostics.png",dpi=110); plt.show()

## 8. Test submission (train+val reference, soft transduction)

In [10]:
ref={k:np.concatenate([pack("train")[k],pack("val")[k]]) for k in pack("train")}
ry=np.concatenate([y_train,y_val])
test_probs = transductive_centroid_probs(ref["bioclip2"], ry, pack("test")["bioclip2"])
_,obs_ids,tk = observation_topk(test_probs,dfs["test"],ks=(10,))
sub=pd.DataFrame({"observationId":obs_ids,"predictions":[" ".join(map(str,r)) for r in tk[:,:10]]})
sub=pd.read_csv("data/FungiCLEF25-SAMPLE_SUBMISSION.csv")[["observationId"]].merge(sub,on="observationId",how="left")
sub.to_csv("submission_ensemble.csv",index=False)
print("wrote submission_ensemble.csv",sub.shape); sub.head()

wrote submission_ensemble.csv (999, 2)


,observationId,predictions
0,4100099350,197 1269 377 349 2384 163 202 1314 271 274
1,4100096393,1624 1549 1555 1213 608 631 1626 1622 1026 1302
2,4100103428,313 1277 2408 1284 1354 1656 1286 142 1290 414
3,4100096438,259 254 250 38 1858 126 1028 1554 326 40
4,4100102708,929 1606 1909 107 2287 2124 914 1267 847 285


## Results

| Configuration | top-1 | top-10 |
|---|---|---|
| Old baselines (ResNet-50 kNN / EfficientNet FT) | ~5–8% | — |
| BioCLIP-2 centroid (train ref) | 46.7% | 75.4% |
| + TTA + train+val reference | ~52.9% | ~81.9% |
| **+ soft transductive prototypes (honest est. test)** | **~53.0%** | **~83.2%** |

A domain foundation model + non-parametric head + folding val into the reference pool
+ soft transduction clears the ~80% bar comfortably — trained in seconds, no backbone
fine-tuning. **Further:** LoRA-finetune BioCLIP-2 with heavy augmentation; 5-crop TTA;
Sinkhorn/optimal-transport label assignment for the transductive step.